# اختبار HAT ×4 على صور الـLow Resolution من Google Drive

هذه الـNotebook تستخدم نفس نموذج **HAT ×4 pretrained** الموجود في الـNotebook السابقة، لكن بدل رفع صورة يدويًا ستقوم بـ:

1. ربط Google Drive.
2. قراءة جميع الصور من:
   ```text
   /content/drive/MyDrive/Super_Resolution_28-07-2026/Law_Resolution/
   ```
3. تشغيل HAT ×4 باستخدام **Tile Inference** لتقليل استهلاك ذاكرة الـGPU.
4. حفظ صورة الـLow Resolution المرئية، ونتيجة Bicubic ×4، ونتيجة HAT ×4 داخل:
   ```text
   /content/drive/MyDrive/Super_Resolution_28-07-2026/HAT_Test_Results/
   ```
5. حفظ ملخص التشغيل في ملف CSV.

## ملاحظة علمية مهمة

ملف `MS_CLIP.tif` يحتوي على **6 Bands**، بينما وزن HAT الجاهز يستقبل **3 قنوات RGB فقط**. لذلك سيتم تكوين صورة RGB مرئية باستخدام:

```text
Band 3 → Red
Band 2 → Green
Band 1 → Blue
```

الناتج هنا **اختبار بصري** لوزن HAT المدرّب مسبقًا، وليس Pan-sharpening علميًا، ولا يحافظ على القيم الإشعاعية الأصلية للـ6 Bands.

## 1) تفعيل GPU

في Google Colab اختر:

```text
Runtime → Change runtime type → T4 GPU
```

In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "GPU غير مفعّل. اختر Runtime → Change runtime type → T4 GPU ثم أعد التشغيل."
    )

PyTorch: 2.11.0+cu128
CUDA Available: True
GPU: Tesla T4


## 2) تنزيل HAT وتثبيت المكتبات

In [2]:
from pathlib import Path
import site
import subprocess
import sys

HAT_DIR = Path("/content/HAT")

if not HAT_DIR.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/XPixelGroup/HAT.git",
            str(HAT_DIR),
        ],
        check=True,
    )
else:
    print("HAT repository already exists:", HAT_DIR)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    cwd=HAT_DIR,
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", "."],
    cwd=HAT_DIR,
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "rasterio",
        "pillow",
        "matplotlib",
        "tqdm",
        "pandas",
        "affine",
    ],
    check=True,
)

# إصلاح توافق BasicSR مع إصدارات torchvision الحديثة في Colab
for site_dir in site.getsitepackages():
    degradation_file = Path(site_dir) / "basicsr/data/degradations.py"

    if not degradation_file.exists():
        continue

    text = degradation_file.read_text(encoding="utf-8")

    old_import = (
        "from torchvision.transforms.functional_tensor "
        "import rgb_to_grayscale"
    )
    new_import = (
        "from torchvision.transforms.functional "
        "import rgb_to_grayscale"
    )

    if old_import in text:
        degradation_file.write_text(
            text.replace(old_import, new_import),
            encoding="utf-8",
        )
        print("Patched:", degradation_file)
    else:
        print("BasicSR compatibility patch already applied.")

    break

print("Installation completed.")

Patched: /usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py
Installation completed.


## 3) تنزيل وزن HAT ×4

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

WEIGHT_DIR = Path("/content/HAT/experiments/pretrained_models")
WEIGHT_DIR.mkdir(parents=True, exist_ok=True)

WEIGHT_PATH = WEIGHT_DIR / "HAT_SRx4_ImageNet-pretrain.pth"

WEIGHT_URL = (
    "https://huggingface.co/Acly/hat/resolve/main/"
    "HAT_SRx4_ImageNet-pretrain.pth"
)

if not WEIGHT_PATH.exists():
    print("Downloading HAT weight...")
    urlretrieve(WEIGHT_URL, WEIGHT_PATH)
else:
    print("Weight already exists:", WEIGHT_PATH)

if not WEIGHT_PATH.exists() or WEIGHT_PATH.stat().st_size < 10_000_000:
    raise RuntimeError("فشل تنزيل وزن HAT أو الملف غير مكتمل.")

print("Weight path:", WEIGHT_PATH)
print("Weight size:", round(WEIGHT_PATH.stat().st_size / 1024**2, 2), "MB")

Weight path: /content/HAT/experiments/pretrained_models/HAT_SRx4_ImageNet-pretrain.pth
Weight size: 81.19 MB


## 4) ربط Google Drive وتحديد مسارات الإدخال والإخراج

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Super_Resolution_28-07-2026"
)

# المسار الأساسي الذي تم إنشاؤه سابقًا.
# تمت إضافة Low_Resolution كبديل فقط في حالة تصحيح اسم الفولدر لاحقًا.
INPUT_CANDIDATES = [
    PROJECT_DIR / "Law_Resolution",
    PROJECT_DIR / "Low_Resolution",
]

INPUT_DIR = next(
    (folder for folder in INPUT_CANDIDATES if folder.exists()),
    INPUT_CANDIDATES[0],
)

OUTPUT_DIR = PROJECT_DIR / "HAT_Test_Results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project folder:", PROJECT_DIR)
print("Input folder:", INPUT_DIR)
print("Output folder:", OUTPUT_DIR)

if not INPUT_DIR.exists():
    raise FileNotFoundError(
        "لم أجد فولدر Law_Resolution أو Low_Resolution داخل المشروع."
    )

## 5) إعدادات الاختبار

- استخدم `RUN_MODE = "crop"` أولًا للتأكد أن كل شيء يعمل بسرعة.
- بعد نجاح التجربة غيّره إلى `RUN_MODE = "full"` لتشغيل الصورة كاملة.
- عند نفاد ذاكرة GPU، يتم تقليل `TILE_SIZE` تلقائيًا من 128 إلى 64.

In [ ]:
RUN_MODE = "crop"       # "crop" للتجربة السريعة، أو "full" للصورة كاملة
CROP_SIZE = 512          # حجم الجزء من صورة LR عند RUN_MODE="crop"

SCALE = 4
WINDOW_SIZE = 16
TILE_SIZE = 128
TILE_PAD = 16

RGB_BANDS = (3, 2, 1)   # Rasterio bands are 1-based
USE_AMP = True
SAVE_GEOTIFF = True

SUPPORTED_EXTENSIONS = {
    ".tif", ".tiff", ".png", ".jpg", ".jpeg"
}

assert RUN_MODE in {"full", "crop"}
assert TILE_SIZE % WINDOW_SIZE == 0
assert TILE_PAD % WINDOW_SIZE == 0
assert all(band >= 1 for band in RGB_BANDS)

print("Run mode:", RUN_MODE)
print("Crop size:", CROP_SIZE)
print("Tile size:", TILE_SIZE)
print("RGB bands:", RGB_BANDS)

## 6) عرض الصور التي سيدخلها النموذج

In [ ]:
input_files = sorted(
    path
    for path in INPUT_DIR.iterdir()
    if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS
)

print("Number of supported images:", len(input_files))

for index, path in enumerate(input_files, start=1):
    print(f"{index:02d}. {path.name}")

if not input_files:
    raise FileNotFoundError(
        f"لا توجد صور مدعومة داخل: {INPUT_DIR}"
    )

## 7) إنشاء نموذج HAT وتحميل الوزن

In [ ]:
import sys
import torch

sys.path.insert(0, "/content/HAT")

from hat.archs.hat_arch import HAT

model = HAT(
    upscale=4,
    in_chans=3,
    img_size=64,
    window_size=16,
    compress_ratio=3,
    squeeze_factor=30,
    conv_scale=0.01,
    overlap_ratio=0.5,
    img_range=1.0,
    depths=[6, 6, 6, 6, 6, 6],
    embed_dim=180,
    num_heads=[6, 6, 6, 6, 6, 6],
    mlp_ratio=2,
    upsampler="pixelshuffle",
    resi_connection="1conv",
)

checkpoint = torch.load(WEIGHT_PATH, map_location="cpu")

if "params_ema" in checkpoint:
    state_dict = checkpoint["params_ema"]
elif "params" in checkpoint:
    state_dict = checkpoint["params"]
else:
    state_dict = checkpoint

model.load_state_dict(state_dict, strict=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device).eval()

print("Device:", device)
print("Model parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
print("Weights loaded successfully.")

## 8) دوال قراءة الصور وتحضير RGB

بالنسبة إلى GeoTIFF متعدد الباندات:

- إذا كان يحتوي على الباندات المطلوبة، يستخدم `3-2-1`.
- إذا كان Band واحدة، يكررها ثلاث مرات.
- يطبق Percentile Stretch من 1% إلى 99% لإنتاج RGB مرئي بين 0 و1.

بالنسبة إلى PNG/JPG، تُستخدم قيم RGB الأصلية مقسومة على 255.

In [ ]:
import numpy as np
import rasterio
from PIL import Image
from rasterio.windows import Window
from rasterio.windows import transform as window_transform

Image.MAX_IMAGE_PIXELS = None


def stretch_band(band, low_percentile=1, high_percentile=99):
    band = band.astype(np.float32)

    valid_mask = np.isfinite(band) & (band > 0)
    valid = band[valid_mask]

    if valid.size == 0:
        return np.zeros_like(band, dtype=np.float32)

    low = np.percentile(valid, low_percentile)
    high = np.percentile(valid, high_percentile)

    if high <= low:
        high = low + 1.0

    stretched = (band - low) / (high - low)
    stretched[~np.isfinite(stretched)] = 0

    return np.clip(stretched, 0, 1).astype(np.float32)


def center_crop_bounds(width, height, crop_size):
    crop_w = min(crop_size, width)
    crop_h = min(crop_size, height)
    crop_x = (width - crop_w) // 2
    crop_y = (height - crop_h) // 2
    return crop_x, crop_y, crop_w, crop_h


def load_input_image(image_path, run_mode="full", crop_size=512):
    suffix = image_path.suffix.lower()

    metadata = {
        "is_geotiff": False,
        "crs": None,
        "transform": None,
        "crop_x": 0,
        "crop_y": 0,
        "source_width": None,
        "source_height": None,
        "source_bands": None,
    }

    if suffix in {".tif", ".tiff"}:
        with rasterio.open(image_path) as src:
            metadata["source_width"] = src.width
            metadata["source_height"] = src.height
            metadata["source_bands"] = src.count

            print(
                f"{image_path.name}: "
                f"{src.count} Bands, {src.width}×{src.height}, "
                f"dtype={src.dtypes[0]}"
            )

            if run_mode == "crop":
                crop_x, crop_y, crop_w, crop_h = center_crop_bounds(
                    src.width, src.height, crop_size
                )
                window = Window(crop_x, crop_y, crop_w, crop_h)
            else:
                crop_x, crop_y = 0, 0
                window = None

            if src.count >= max(RGB_BANDS):
                band_indexes = list(RGB_BANDS)
                data = src.read(band_indexes, window=window)
            elif src.count == 1:
                single = src.read(1, window=window)
                data = np.stack([single, single, single], axis=0)
            elif src.count >= 3:
                band_indexes = [1, 2, 3]
                data = src.read(band_indexes, window=window)
            else:
                raise ValueError(
                    f"Unsupported band count in {image_path.name}: {src.count}"
                )

            selected_transform = (
                window_transform(window, src.transform)
                if window is not None
                else src.transform
            )

            metadata.update({
                "is_geotiff": True,
                "crs": src.crs,
                "transform": selected_transform,
                "crop_x": crop_x,
                "crop_y": crop_y,
            })

        normalized = np.stack(
            [stretch_band(data[channel]) for channel in range(3)],
            axis=0,
        )

    else:
        pil_image = Image.open(image_path).convert("RGB")
        metadata["source_width"], metadata["source_height"] = pil_image.size
        metadata["source_bands"] = 3

        if run_mode == "crop":
            crop_x, crop_y, crop_w, crop_h = center_crop_bounds(
                pil_image.width, pil_image.height, crop_size
            )
            pil_image = pil_image.crop(
                (crop_x, crop_y, crop_x + crop_w, crop_y + crop_h)
            )
            metadata["crop_x"] = crop_x
            metadata["crop_y"] = crop_y

        normalized = (
            np.asarray(pil_image, dtype=np.float32) / 255.0
        )
        normalized = np.moveaxis(normalized, -1, 0)

    return np.ascontiguousarray(normalized, dtype=np.float32), metadata

## 9) دالة Tile Inference

In [ ]:
import torch.nn.functional as F
from tqdm.auto import tqdm


def pad_to_window(tensor, window_size=16):
    height, width = tensor.shape[-2:]

    pad_h = (window_size - height % window_size) % window_size
    pad_w = (window_size - width % window_size) % window_size

    pad_mode = (
        "reflect"
        if height > pad_h and width > pad_w
        else "replicate"
    )

    padded = F.pad(
        tensor,
        (0, pad_w, 0, pad_h),
        mode=pad_mode,
    )

    return padded


@torch.inference_mode()
def run_hat_tiled(
    rgb_chw,
    model,
    device,
    scale=4,
    tile_size=128,
    tile_pad=16,
    window_size=16,
    use_amp=True,
):
    channels, height, width = rgb_chw.shape

    output = np.zeros(
        (height * scale, width * scale, channels),
        dtype=np.uint8,
    )

    x_positions = list(range(0, width, tile_size))
    y_positions = list(range(0, height, tile_size))

    progress = tqdm(
        total=len(x_positions) * len(y_positions),
        desc=f"HAT ×{scale}",
    )

    for y0 in y_positions:
        for x0 in x_positions:
            x1 = min(x0 + tile_size, width)
            y1 = min(y0 + tile_size, height)

            context_x0 = max(0, x0 - tile_pad)
            context_y0 = max(0, y0 - tile_pad)
            context_x1 = min(width, x1 + tile_pad)
            context_y1 = min(height, y1 + tile_pad)

            tile_numpy = rgb_chw[
                :,
                context_y0:context_y1,
                context_x0:context_x1,
            ]

            tile_tensor = (
                torch.from_numpy(tile_numpy)
                .unsqueeze(0)
                .to(device=device, dtype=torch.float32, non_blocking=True)
            )

            tile_tensor = pad_to_window(tile_tensor, window_size)

            amp_enabled = use_amp and device.type == "cuda"

            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=amp_enabled,
            ):
                tile_output = model(tile_tensor).clamp_(0, 1)

            context_h = context_y1 - context_y0
            context_w = context_x1 - context_x0

            tile_output = tile_output[
                :,
                :,
                :context_h * scale,
                :context_w * scale,
            ]

            core_x0 = (x0 - context_x0) * scale
            core_y0 = (y0 - context_y0) * scale
            core_x1 = core_x0 + (x1 - x0) * scale
            core_y1 = core_y0 + (y1 - y0) * scale

            core = tile_output[
                0,
                :,
                core_y0:core_y1,
                core_x0:core_x1,
            ]

            core_uint8 = (
                core.permute(1, 2, 0)
                .float()
                .cpu()
                .numpy()
            )
            core_uint8 = np.clip(
                np.rint(core_uint8 * 255),
                0,
                255,
            ).astype(np.uint8)

            output[
                y0 * scale:y1 * scale,
                x0 * scale:x1 * scale,
            ] = core_uint8

            del tile_tensor, tile_output, core
            progress.update(1)

    progress.close()
    return output

## 10) تشغيل الاختبار على جميع صور فولدر Low Resolution

النوتبوك تحفظ لكل صورة:

- `*_LR_RGB_*.png`
- `*_Bicubic_x4_*.png`
- `*_HAT_x4_*.png`
- `*_HAT_x4_*.tif` للصورة الجغرافية
- ملف `HAT_test_summary_*.csv`

In [ ]:
import gc
import time
import pandas as pd
from affine import Affine

summary_rows = []
last_result = None

for image_index, image_path in enumerate(input_files, start=1):
    print("\n" + "=" * 80)
    print(f"Processing {image_index}/{len(input_files)}: {image_path.name}")

    rgb_input, metadata = load_input_image(
        image_path,
        run_mode=RUN_MODE,
        crop_size=CROP_SIZE,
    )

    print("Model input shape (C,H,W):", rgb_input.shape)

    start_time = time.time()
    used_tile_size = TILE_SIZE

    try:
        sr_rgb = run_hat_tiled(
            rgb_chw=rgb_input,
            model=model,
            device=device,
            scale=SCALE,
            tile_size=used_tile_size,
            tile_pad=TILE_PAD,
            window_size=WINDOW_SIZE,
            use_amp=USE_AMP,
        )

    except torch.cuda.OutOfMemoryError:
        print("GPU OOM: retrying with TILE_SIZE=64.")

        gc.collect()
        torch.cuda.empty_cache()

        used_tile_size = 64
        sr_rgb = run_hat_tiled(
            rgb_chw=rgb_input,
            model=model,
            device=device,
            scale=SCALE,
            tile_size=used_tile_size,
            tile_pad=TILE_PAD,
            window_size=WINDOW_SIZE,
            use_amp=USE_AMP,
        )

    elapsed = time.time() - start_time

    lr_uint8 = np.clip(
        np.rint(np.moveaxis(rgb_input, 0, -1) * 255),
        0,
        255,
    ).astype(np.uint8)

    bicubic = np.asarray(
        Image.fromarray(lr_uint8).resize(
            (sr_rgb.shape[1], sr_rgb.shape[0]),
            Image.Resampling.BICUBIC,
        )
    )

    suffix = "full" if RUN_MODE == "full" else "crop"
    base_name = image_path.stem

    lr_png_path = OUTPUT_DIR / f"{base_name}_LR_RGB_{suffix}.png"
    bicubic_png_path = OUTPUT_DIR / f"{base_name}_Bicubic_x4_{suffix}.png"
    hat_png_path = OUTPUT_DIR / f"{base_name}_HAT_x4_{suffix}.png"

    Image.fromarray(lr_uint8).save(lr_png_path)
    Image.fromarray(bicubic).save(bicubic_png_path)
    Image.fromarray(sr_rgb).save(hat_png_path)

    hat_tif_path = None

    if metadata["is_geotiff"] and SAVE_GEOTIFF:
        hat_tif_path = OUTPUT_DIR / f"{base_name}_HAT_x4_{suffix}.tif"

        output_transform = metadata["transform"] * Affine.scale(
            1 / SCALE,
            1 / SCALE,
        )

        output_profile = {
            "driver": "GTiff",
            "height": sr_rgb.shape[0],
            "width": sr_rgb.shape[1],
            "count": 3,
            "dtype": "uint8",
            "crs": metadata["crs"],
            "transform": output_transform,
            "compress": "deflate",
            "predictor": 2,
            "interleave": "pixel",
            "photometric": "RGB",
            "BIGTIFF": "IF_SAFER",
        }

        with rasterio.open(hat_tif_path, "w", **output_profile) as dst:
            dst.write(np.moveaxis(sr_rgb, -1, 0))

    summary_rows.append({
        "input_file": image_path.name,
        "run_mode": RUN_MODE,
        "source_bands": metadata["source_bands"],
        "source_width": metadata["source_width"],
        "source_height": metadata["source_height"],
        "model_input_width": rgb_input.shape[2],
        "model_input_height": rgb_input.shape[1],
        "output_width": sr_rgb.shape[1],
        "output_height": sr_rgb.shape[0],
        "scale": SCALE,
        "rgb_bands": str(RGB_BANDS),
        "tile_size": used_tile_size,
        "elapsed_seconds": round(elapsed, 2),
        "lr_rgb_png": str(lr_png_path),
        "bicubic_png": str(bicubic_png_path),
        "hat_png": str(hat_png_path),
        "hat_geotiff": str(hat_tif_path) if hat_tif_path else "",
    })

    last_result = {
        "image_name": image_path.name,
        "lr": lr_uint8,
        "bicubic": bicubic,
        "hat": sr_rgb,
    }

    print("Output shape (H,W,C):", sr_rgb.shape)
    print("Time:", round(elapsed / 60, 2), "minutes")
    print("Saved:", hat_png_path)

    if hat_tif_path:
        print("Saved GeoTIFF:", hat_tif_path)

    del rgb_input
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

summary_df = pd.DataFrame(summary_rows)
summary_csv_path = OUTPUT_DIR / f"HAT_test_summary_{RUN_MODE}.csv"
summary_df.to_csv(summary_csv_path, index=False)

print("\nAll images completed.")
print("Summary CSV:", summary_csv_path)

display(summary_df)

## 11) عرض آخر نتيجة: LR مقابل Bicubic ×4 مقابل HAT ×4

In [ ]:
import matplotlib.pyplot as plt


def make_preview(image, max_side=1400):
    height, width = image.shape[:2]
    ratio = max(height, width) / max_side

    if ratio <= 1:
        return image

    new_width = max(1, round(width / ratio))
    new_height = max(1, round(height / ratio))

    return np.asarray(
        Image.fromarray(image).resize(
            (new_width, new_height),
            Image.Resampling.LANCZOS,
        )
    )


if last_result is None:
    raise RuntimeError("لم يتم تشغيل أي صورة بعد.")

lr_preview = make_preview(last_result["lr"])
bicubic_preview = make_preview(last_result["bicubic"])
hat_preview = make_preview(last_result["hat"])

fig, axes = plt.subplots(1, 3, figsize=(24, 9))

axes[0].imshow(lr_preview)
axes[0].set_title("Original Low Resolution RGB 3-2-1")
axes[0].axis("off")

axes[1].imshow(bicubic_preview)
axes[1].set_title("Bicubic ×4")
axes[1].axis("off")

axes[2].imshow(hat_preview)
axes[2].set_title("Pretrained HAT ×4")
axes[2].axis("off")

plt.suptitle(last_result["image_name"], fontsize=16)
plt.tight_layout()
plt.show()

## 12) مقارنة نفس المنطقة بالتكبير

الإحداثيات التالية تخص صورتي Bicubic وHAT بعد التكبير ×4. يمكنك تعديل `ZOOM_X`, `ZOOM_Y`, و`ZOOM_SIZE`.

In [ ]:
hat_image = last_result["hat"]
bicubic_image = last_result["bicubic"]

output_height, output_width = hat_image.shape[:2]

ZOOM_SIZE = min(1000, output_height, output_width)
ZOOM_X = (output_width - ZOOM_SIZE) // 2
ZOOM_Y = (output_height - ZOOM_SIZE) // 2

hat_zoom = hat_image[
    ZOOM_Y:ZOOM_Y + ZOOM_SIZE,
    ZOOM_X:ZOOM_X + ZOOM_SIZE,
]

bicubic_zoom = bicubic_image[
    ZOOM_Y:ZOOM_Y + ZOOM_SIZE,
    ZOOM_X:ZOOM_X + ZOOM_SIZE,
]

fig, axes = plt.subplots(1, 2, figsize=(18, 9))

axes[0].imshow(bicubic_zoom)
axes[0].set_title("Bicubic ×4 — Zoom")
axes[0].axis("off")

axes[1].imshow(hat_zoom)
axes[1].set_title("HAT ×4 — Same Zoom")
axes[1].axis("off")

plt.tight_layout()
plt.show()

print("Zoom X:", ZOOM_X)
print("Zoom Y:", ZOOM_Y)
print("Zoom size:", ZOOM_SIZE)

# قراءة النتيجة

بعد انتهاء التشغيل ستجد النتائج داخل:

```text
/content/drive/MyDrive/Super_Resolution_28-07-2026/HAT_Test_Results/
```

ابدأ بـ:

```python
RUN_MODE = "crop"
```

وبعد التأكد من نجاح الاختبار غيّرها إلى:

```python
RUN_MODE = "full"
```

## لماذا لا نحسب PSNR وSSIM هنا؟

لا توجد صورة **RGB High-Resolution Ground Truth** مطابقة لصورة `MS_CLIP.tif`. لذلك حساب PSNR أو SSIM بين HAT والصورة المتاحة لن يكون تقييمًا صحيحًا. المقارنة الحالية هي:

```text
Low Resolution RGB 3-2-1
vs
Bicubic ×4
vs
Pretrained HAT ×4
```

كما أن ملف GeoTIFF الناتج يمثل **RGB مرئيًا بعمق 8-bit بعد Stretch**، وليس استعادة علمية للباندات الست أو لقيمها الأصلية.